In [1]:
import sys

sys.path.append("..")

from common.metrics import metrics
from common.preprocessing import preprocessing_cn7
from common.train_test_split import split_data
from common.model_composition import model_composition

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 데이터 불러오기
train_cn7 = pd.read_csv(r"C:\Users\Administrator\Desktop\kamp\1. 사출성형기 AI 데이터셋\1. 사출성형기 AI 데이터셋\moldset_labeled_cn7.csv")

In [3]:
X_cn7, y_cn7 = preprocessing_cn7(train_cn7)

print("X shape:", X_cn7.shape)
print("y shape:", y_cn7.shape)

X shape: (1211, 23)
y shape: (1211,)


In [4]:
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import average_precision_score


# 재현성
np.random.seed(42)
torch.manual_seed(42)


X_array = X_cn7.to_numpy(dtype=np.float32)
y_array = y_cn7.to_numpy()

input_dim = X_array.shape[1]

print("Input Dimension:", input_dim)


class AutoEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8)
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim)
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_ae_score = np.zeros(len(X_array))


for fold, (train_idx, val_idx) in enumerate(
    cv.split(X_array, y_array),
    start=1
):

    print(f"\nFold {fold}")

    X_train = X_array[train_idx]
    y_train = y_array[train_idx]
    X_val = X_array[val_idx]

    # 정상 데이터만 AutoEncoder 학습
    X_train_normal = X_train[y_train == 0]

    train_tensor = torch.tensor(
        X_train_normal,
        dtype=torch.float32
    )

    train_loader = DataLoader(
        TensorDataset(train_tensor),
        batch_size=64,
        shuffle=True
    )

    model = AutoEncoder(input_dim)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001
    )

    criterion = nn.MSELoss()

    # 학습
    model.train()

    for epoch in range(100):

        for (batch,) in train_loader:

            optimizer.zero_grad()

            reconstructed = model(batch)

            loss = criterion(
                reconstructed,
                batch
            )

            loss.backward()
            optimizer.step()

    # Validation reconstruction error
    model.eval()

    X_val_tensor = torch.tensor(
        X_val,
        dtype=torch.float32
    )

    with torch.no_grad():

        reconstructed = model(
            X_val_tensor
        )

        reconstruction_error = torch.mean(
            (X_val_tensor - reconstructed) ** 2,
            dim=1
        )

    oof_ae_score[val_idx] = (
        reconstruction_error.numpy()
    )


print(
    "\nAutoEncoder OOF PR-AUC:",
    average_precision_score(
        y_array,
        oof_ae_score
    )
)

Input Dimension: 23

Fold 1

Fold 2

Fold 3

Fold 4

Fold 5

AutoEncoder OOF PR-AUC: 0.3962270560613178


In [6]:
ae_df = pd.DataFrame({
    "Target": y_array,
    "AE_Reconstruction_Error": oof_ae_score
})

ae_df["Rank"] = (
    ae_df["AE_Reconstruction_Error"]
    .rank(
        ascending=False,
        method="min"
    )
)

display(
    ae_df[
        ae_df["Target"] == 1
    ].sort_values("Rank")
)

,Target,AE_Reconstruction_Error,Rank
119,1,6.336295,1.0
117,1,6.216015,2.0
120,1,5.927997,3.0
118,1,5.895386,4.0
115,1,5.719362,5.0
116,1,5.696852,6.0
113,1,0.514666,24.0
105,1,0.318946,106.0
111,1,0.280412,150.0
107,1,0.260222,176.0


In [7]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

results = []

for top_n in [
    5, 6, 10, 15, 20, 25, 30,
    40, 50, 75, 100, 150, 200
]:

    threshold = np.sort(oof_ae_score)[-top_n]

    y_pred = (
        oof_ae_score >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_array,
        y_pred
    ).ravel()

    results.append({
        "Top N": top_n,
        "Threshold": threshold,
        "Precision": precision_score(
            y_array, y_pred,
            zero_division=0
        ),
        "Recall": recall_score(
            y_array, y_pred,
            zero_division=0
        ),
        "F1": f1_score(
            y_array, y_pred,
            zero_division=0
        ),
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })


ae_threshold_df = pd.DataFrame(results)

display(ae_threshold_df)

,Top N,Threshold,Precision,Recall,F1,TP,FP,FN,TN
0,5,5.719362,1.000000,0.294118,0.454545,5,0,12,1194
1,6,5.696852,1.000000,0.352941,0.521739,6,0,11,1194
2,10,2.135999,0.600000,0.352941,0.444444,6,4,11,1190
3,15,0.628673,0.400000,0.352941,0.375000,6,9,11,1185
4,20,0.559354,0.300000,0.352941,0.324324,6,14,11,1180
5,25,0.492042,0.280000,0.411765,0.333333,7,18,10,1176
6,30,0.475607,0.233333,0.411765,0.297872,7,23,10,1171
7,40,0.416382,0.175000,0.411765,0.245614,7,33,10,1161
8,50,0.391461,0.140000,0.411765,0.208955,7,43,10,1151
9,75,0.352596,0.093333,0.411765,0.152174,7,68,10,1126
